In [17]:
#### ------------------------------------------------------------------------------------------
#### author: Ranjan Barman, date: July 24, 2025 (modified for POST_NAT_BRCA)
#### Mapped POST_NAT HoverNet Immune-Only NPIFs (all tiles) to BRCA subtype status
#### --------------------------------------------------------------------------------------------

import os
import pandas as pd

# Set working directory
_wpath_ = "/data/Lab_ruppin/Ranjan/HnE/"
os.chdir(_wpath_)
print(f"Working directory: {_wpath_}\n")

# Dataset name
dataset_name = "POST_NAT_BRCA"

# File paths
npif_file = f"{dataset_name}/HoverNet/outputs/POST_NAT_BRCA_HoverNet_ImmuneOnly_NPIFs.csv"
slide_list_file = "/data/Ruppin_AI/Datasets/Post_NAT_BRCA/processed/Post_NAT_BRCA_slide_list.tsv"
clinical_metadata_file = "/data/Ruppin_AI/Datasets/Post_NAT_BRCA/processed/Post_NAT_BRCA_clinical_metadata_short.tsv"

# Load NPIFs and extract Slide_ID
npif_df = pd.read_csv(npif_file)
npif_df["Slide_ID"] = npif_df["Slide_ID"].astype(int)

# Load slide-to-patient mapping and clinical metadata
slide_list_df = pd.read_csv(slide_list_file, sep="\t")
clinical_metadata_df = pd.read_csv(clinical_metadata_file, sep="\t")
slide_list_df
clinical_metadata_df


Working directory: /data/Lab_ruppin/Ranjan/HnE/



,Patient_ID,Age,Menopausal_status,Lymphovascular_invasion,Histology_type,Histology_grade,HER2_status,ER_status,PR_status,Clinical_subtype,Clinical_subtype_fine,NAT_regimen,Surgery_type_breast,Surgery_type_LN,Response
0,P1,67,Post,1.0,IDC,3,0.0,1.0,1.0,HR+,HR+,Chemo+Endocrine,Total mastectomy,Axillary LN dissection,PDR
1,P2,30,Pre,1.0,IDC,3,1.0,0.0,0.0,HER2+,HER2+,Chemo+Anti-HER2,Total mastectomy,Axillary LN dissection,PDR
2,P3,65,Post,1.0,IDC,2,0.0,1.0,1.0,HR+,HR+,Chemo,Partial mastectomy/lumpectomy,Axillary LN dissection,PDR
3,P4,41,Pre,0.0,IDC,2,0.0,1.0,1.0,HR+,HR+,Chemo,Total mastectomy,Sentinel LN biopsy,NDR
4,P5,47,Pre,0.0,IDC,Undetermined,1.0,1.0,1.0,HER2+,TPBC,Chemo+Anti-HER2,Total mastectomy,Axillary LN dissection,PDR
5,P6,58,Post,1.0,ILC,2,0.0,1.0,1.0,HR+,HR+,Chemo,Total mastectomy,Axillary LN dissection,PDR
6,P7,39,Pre,0.0,IDC,3,0.0,0.0,0.0,TNBC,TNBC,Chemo,Total mastectomy,Sentinel LN biopsy,PDR
7,P8,52,Pre,0.0,IDC,2,0.0,1.0,1.0,HR+,HR+,Chemo,Total mastectomy,Axillary LN dissection,PDR
8,P9,34,Pre,0.0,IDC,2,1.0,1.0,0.0,HER2+,TPBC,Chemo+Anti-HER2,Total mastectomy,Axillary LN dissection,PDR
9,P10,55,Post,1.0,IDC,2,0.0,1.0,1.0,HR+,HR+,Chemo,Total mastectomy,Axillary LN dissection,PDR


In [18]:
# Merge Slide_ID to get Patient_ID
npif_mapped_df = pd.merge(npif_df, slide_list_df[["Patient_ID", "Slide_ID"]], on="Slide_ID", how="left")
npif_mapped_df = npif_mapped_df.dropna(subset=["Patient_ID"])
npif_mapped_df["Patient_ID"] = npif_mapped_df["Patient_ID"].astype(str)

# Drop any pre-existing clinical columns to avoid duplication
columns_to_drop = ["HER2_Status", "ER_Status", "PR_Status", "Clinical_subtype"]
npif_mapped_df = npif_mapped_df.drop(columns=[col for col in columns_to_drop if col in npif_mapped_df.columns])

# Prepare and rename clinical subtype columns
clinical_metadata_df["Patient_ID"] = clinical_metadata_df["Patient_ID"].astype(str)
subtypes_df = clinical_metadata_df.rename(columns={
    "HER2_status": "HER2_Status",
    "ER_status": "ER_Status",
    "PR_status": "PR_Status"
})[["Patient_ID", "HER2_Status", "ER_Status", "PR_Status", "Clinical_subtype"]]

# Map binary values to 'Positive'/'Negative'
binary_map = {1.0: "Positive", 0.0: "Negative"}
subtypes_df["HER2_Status"] = subtypes_df["HER2_Status"].map(binary_map)
subtypes_df["ER_Status"] = subtypes_df["ER_Status"].map(binary_map)
subtypes_df["PR_Status"] = subtypes_df["PR_Status"].map(binary_map)

# Merge subtype info into npif_mapped_df
merged_df = pd.merge(subtypes_df, npif_mapped_df, on="Patient_ID", how="inner")

# Reorder to make Patient_ID the first column
cols = merged_df.columns.tolist()
cols.insert(0, cols.pop(cols.index("Patient_ID")))
merged_df = merged_df[cols]

# Print unique Slide_Name and Patient_ID values
print("Patient_IDs from POST_NAT_BRCA_HoverNet_NPIFs (Total: {}):".format(len(merged_df)))
print(merged_df["Patient_ID"].unique())

print("\nPatient IDs from POST_NAT clinical metadata (Total: {}):".format(len(subtypes_df)))
print(subtypes_df["Patient_ID"].unique())

# Find non-matching values
slide_ids = set(merged_df["Patient_ID"])
patient_ids = set(subtypes_df["Patient_ID"])

non_matching_slides = slide_ids - patient_ids
non_matching_patients = patient_ids - slide_ids

print("\nHoverNet NPIFs that do NOT have a matching Patient_ID:")
print(non_matching_slides)

print("\nClinical metadata entries that do NOT have a matching Slide NPIF:")
print(non_matching_patients)



Patient_IDs from POST_NAT_BRCA_HoverNet_NPIFs (Total: 93):
['P1' 'P2' 'P3' 'P4' 'P5' 'P6' 'P7' 'P8' 'P9' 'P10' 'P11' 'P12' 'P13'
 'P14' 'P15' 'P16' 'P17' 'P18' 'P19' 'P20' 'P21' 'P22' 'P23' 'P24' 'P25'
 'P26' 'P27' 'P28' 'P29' 'P30' 'P31' 'P32' 'P33' 'P34' 'P35' 'P37' 'P38'
 'P39' 'P40' 'P41' 'P42' 'P43' 'P44' 'P45' 'P46' 'P47' 'P48' 'P49' 'P50'
 'P51' 'P52' 'P53' 'P54']

Patient IDs from POST_NAT clinical metadata (Total: 54):
['P1' 'P2' 'P3' 'P4' 'P5' 'P6' 'P7' 'P8' 'P9' 'P10' 'P11' 'P12' 'P13'
 'P14' 'P15' 'P16' 'P17' 'P18' 'P19' 'P20' 'P21' 'P22' 'P23' 'P24' 'P25'
 'P26' 'P27' 'P28' 'P29' 'P30' 'P31' 'P32' 'P33' 'P34' 'P35' 'P36' 'P37'
 'P38' 'P39' 'P40' 'P41' 'P42' 'P43' 'P44' 'P45' 'P46' 'P47' 'P48' 'P49'
 'P50' 'P51' 'P52' 'P53' 'P54']

HoverNet NPIFs that do NOT have a matching Patient_ID:
set()

Clinical metadata entries that do NOT have a matching Slide NPIF:
{'P36'}


In [19]:
# Save merged (slide-level) result
output_dir = f"{dataset_name}/outputs/HoverNet/Subtypes/"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "HoverNet_ImmuneOnly_NPIFs_All_Tiles_POST_NAT_BRCA_Mapped_BRCA_Status.csv")
# --------------------------------------
# Patient-level aggregation of NPIFs
# --------------------------------------

# Identify NPIF feature columns to average
mean_std_cols = [col for col in merged_df.columns if col.startswith("Mean ") or col.startswith("Std ")]

# Aggregate NPIFs by Patient_ID (mean), keep first for clinical/status columns
patient_level_df = merged_df.groupby("Patient_ID").agg({
    "HER2_Status": "first",
    "ER_Status": "first",
    "PR_Status": "first",
    "Clinical_subtype": "first",
    **{col: "mean" for col in mean_std_cols}
}).reset_index()

# Save patient-level aggregated results to the same file (overwrite)
patient_level_df.to_csv(output_file, index=False)
print(f"\nPatient-level averaged data saved to: {output_file}")
patient_level_df



Patient-level averaged data saved to: POST_NAT_BRCA/outputs/HoverNet/Subtypes/HoverNet_ImmuneOnly_NPIFs_All_Tiles_POST_NAT_BRCA_Mapped_BRCA_Status.csv


,Patient_ID,HER2_Status,ER_Status,PR_Status,Clinical_subtype,Mean Area,Mean Major Axis,Mean Minor Axis,Mean Perimeter,Mean Eccentricity,Mean Circularity,Std Area,Std Major Axis,Std Minor Axis,Std Perimeter,Std Eccentricity,Std Circularity
0,P1,Negative,Positive,Positive,HR+,5.680922,3.263634,2.376652,9.342079,0.640795,0.801047,2.126056,0.684666,0.426367,1.806510,0.149244,0.068583
1,P10,Negative,Positive,Positive,HR+,5.893770,3.336161,2.400254,9.515610,0.649265,0.796498,2.447113,0.770223,0.467471,2.039873,0.148196,0.069349
2,P11,Negative,Negative,Negative,TNBC,6.778583,3.510195,2.609509,10.148907,0.624076,0.809013,2.461946,0.732774,0.449556,1.909252,0.148355,0.060563
3,P12,Negative,Negative,Negative,TNBC,6.801173,3.607990,2.545226,10.230945,0.660885,0.795050,2.755116,0.844042,0.481877,2.142935,0.151199,0.068087
4,P13,Negative,Negative,Negative,TNBC,7.986337,3.784082,2.778534,10.954229,0.639125,0.791578,4.145479,0.974017,0.721297,2.785051,0.147258,0.066604
5,P14,Negative,Positive,Positive,HR+,6.239375,3.445621,2.460010,9.813096,0.656056,0.794231,2.423127,0.760337,0.462933,1.992362,0.146668,0.067512
6,P15,Negative,Positive,Positive,HR+,6.457001,3.526905,2.477085,9.992821,0.668401,0.790029,2.681561,0.822846,0.497472,2.167903,0.145152,0.069314
7,P16,Negative,Positive,Positive,HR+,5.781735,3.307597,2.381659,9.423323,0.644420,0.800622,2.275362,0.762955,0.429203,1.954134,0.152395,0.069323
8,P17,Negative,Positive,Positive,HR+,5.928892,3.356636,2.404426,9.565747,0.651870,0.793482,2.375010,0.756895,0.471440,1.998427,0.149832,0.070749
9,P18,Negative,Negative,Negative,TNBC,5.518564,3.246748,2.324519,9.239448,0.651114,0.797479,2.017974,0.706341,0.405652,1.825018,0.150332,0.071618
